In [ ]:
# These are default values. Papermill will overwrite these at runtime.
question = "What are the prerequisites for CPT S 223?"
student_context = {"major": "Computer Science", "completed_courses": [], "credits_completed": 0}
use_rag = True


In [ ]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

base_dir = Path("/Volumes/asus/VC-n8n-440")
sys.path.insert(0, str(base_dir / "prompt-search" / "src"))
load_dotenv(base_dir / "prompt-search" / ".env")

index_dir = str(base_dir / "prompt-search" / "data" / "domain")


In [ ]:
import json, os, time
from retrieval.retriever import CourseRetriever
from retrieval.context_builder import ContextBuilder
from llm.claude_client import ClaudeClient
from llm.cache import ResponseCache

start_time = time.time()

# Papermill passes dicts as JSON strings
if isinstance(student_context, str):
    student_context = json.loads(student_context)

retriever = CourseRetriever(index_dir=index_dir)
builder   = ContextBuilder(retriever, top_k=5)
cache     = ResponseCache(db_path=str(base_dir / "prompt-search" / "data" / "cache" / "responses.db"))
client    = ClaudeClient(model="claude-haiku-4-5", api_key=os.getenv("ANTHROPIC_API_KEY"))

# Classify question: only inject student context for degree/ucore questions
_q = question.lower()
_DEGREE_KEYWORDS = {"degree", "graduation", "graduate", "credits remaining", "credits completed",
                    "how many credits", "requirements left", "finish my degree", "on track",
                    "semester remaining", "semesters left", "progress"}
_UCORE_KEYWORDS  = {"ucore", "general education", "gen ed", "core requirement", "writing in major"}
_is_degree_q = any(kw in _q for kw in _DEGREE_KEYWORDS)
_is_ucore_q  = any(kw in _q for kw in _UCORE_KEYWORDS)
_needs_student_ctx = _is_degree_q or _is_ucore_q

# Build student context block — only injected when question type warrants it
student_block = ""
if _needs_student_ctx:
    completed  = student_context.get("completed_courses", [])
    credits_done = student_context.get("credits_completed", 0)
    major      = student_context.get("major", "")
    if completed:
        student_block += f"Student completed courses: {', '.join(completed)}\n"
    if credits_done:
        student_block += f"Credits completed: {credits_done}\n"
    if major and major != "Undeclared":
        student_block += f"Student major: {major}\n"
# If no transcript uploaded or question is prerequisite/schedule, student_block="" → pure RAG

# Build prompt via RAG
prompt, sources = builder.build(question, base_prompt=student_block)

# Call Claude (with cache)
cached = cache.get(prompt, model=client.model, temperature=0.0)
if cached:
    answer = cached
else:
    answer = client.generate(prompt, temperature=0.0, max_tokens=400)
    cache.set(prompt, model=client.model, temperature=0.0, response=answer)

source_codes = [s["course_code"] for s in sources]

output = {
    "answer": answer,
    "sources": source_codes,
    "metadata": {
        "used_rag": use_rag,
        "latency_seconds": round(time.time() - start_time, 2)
    }
}

import tempfile
temp_path = os.path.join(tempfile.gettempdir(), "llm_response.json")
with open(temp_path, "w") as f:
    json.dump(output, f)
print(json.dumps(output))
